# READ ME! 

The following code calculates the [Jensen-Shannon Distance] and [Information rate]. The packages used for the calculation and figures are as followed: 
<ol>
    <li>numpy (numpy 1.25.2)</li>
    <li>scipy (scipy 1.13.1)</li>
    <li>pandas (pandas 1.5.2)</li>
    <li>mne (MNE-Python 1.8.0)</li>
    <li>bids (PyBIDS 0.16.3)</li>
    <li>scikit_posthocs (scikit-posthocs 0.9.0)</li>
    <li>plotly (plotly 5.18.0)</li>
    <li>matplotlib (matplotlib 3.9.2)</li>
    <li>joblib (joblib 1.1.0)</li>
    <li><b>info_geo_plos (package made by author)</b></li>
</ol>

The data used for the analysis is available in OpenNeuro [https://openneuro.org/datasets/ds004504/versions/1.0.8]

The code divided into 3 parts: [Analysis], [Jensen-Shannon Distance], and [information rate calculation]

In [Analysis] section, the the raw data is read using [bids] and [mne]. All the information of the participants is shown in the table consists of [participant_id], [Gender], [Age], [Group], and [MMSE]. The data is grouped into 3 groups for later calculation: [alz_eeg_file] for Alzheimer, [con_eeg_file] for control (healthy), [dem_eeg_file] for frontotemporal dementia. Note that, all the file is in .set format and [mne] is used to read the file. There is a small section to display the signals in [graph:plotting the signals]. <b>User needs to run the [extract the spreadsheet of the participants] to extract [alz_eeg_file], [con_eeg_file], and [dem_eeg_file].</b>

In [Jensen-Shannon Distance], the Jensen-Shannon distance is calculated for all the participants and saved in .tsv file along with the figures. <b>The storing is needed for the next part in [combine all three], [statistical test], and [significant plot for Kruskal-Wallies test].</b> Next, for the [statistical test] the user requires to input the channels and the frequency range for the analysis. 

In [information rate calculation], the information rate is calculated for all the combinations and frequency bands. <b>All the calculation will be saved in .tsv file. Note that, the calculation will take a very long time. Hence, the saving is very necessary.</b> For the [graph: PDF for the information rate over time for each group with standard deviationgraph: PDF for the information rate over time for each group with standard deviation], user requires to change the range of the distribution for better visualization. 

# libraries

In [ ]:
import numpy as np
import pandas as pd
import mne
import os
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
import scikit_posthocs as sp
import info_geo_plos as ig # import library from the info_geo. 

from numpy.lib.stride_tricks import sliding_window_view
from tqdm import tqdm
from bids import BIDSLayout
from bids.tests import get_test_data_path
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy
from scipy.stats import kruskal
from plotly.subplots import make_subplots
from joblib import Parallel, delayed

# Analysis

## extract the spreadsheet of the participants

In [ ]:
# @title Extract the data
#extract the data for [difference group of participants]
data_folder = '/home/hengjie/Desktop/alzheimer_openneuro' #folder where the data is stored. 
participants_file = f'{data_folder}/participants.tsv' #file path for the spreadsheet

participants_df = pd.read_csv(participants_file, delimiter='\t')

#alzheimer group
alz_part = participants_df[participants_df['Group'] == 'A']
alz_id = alz_part['participant_id'].tolist()
alz_id = [s.replace('sub-', '') for s in alz_id]
alz_id = None if len(alz_id)==0 else alz_id

#frontotemporal dementia group
dem_part = participants_df[participants_df['Group'] == 'F']
dem_id = dem_part['participant_id'].tolist()
dem_id = [s.replace('sub-', '') for s in dem_id]
dem_id = None if len(dem_id)==0 else dem_id

#healthy group
con_part = participants_df[participants_df['Group'] == 'C']
con_id = con_part['participant_id'].tolist()
con_id = [s.replace('sub-', '') for s in con_id]
con_id = None if len(con_id)==0 else con_id


#particiapants data
print(participants_df)

In [ ]:
# @title Read the BIDS format dataset
#read the BIDS format dataset
#NOTE: the [derivatives] should use the [derivatives] way of reading it but it does not work in this case. Hence, the author copy the [metadata] to the [derivatives].
data_path = os.path.join(get_test_data_path(), f'{data_folder}/derivatives')
layout = BIDSLayout(data_path)
print(layout)

In [ ]:
file_format = '.set'
alz_eeg_file = layout.get(subject=alz_id, return_type='filename', extension=f'{file_format}')
dem_eeg_file = layout.get(subject=dem_id, return_type='filename', extension=f'{file_format}')
con_eeg_file = layout.get(subject=con_id, return_type='filename', extension=f'{file_format}')

print('number of files for each group')
print(f'alz_eeg_file: \t\t{len(alz_eeg_file)}')
print(f'dem_eeg_file: \t\t{len(dem_eeg_file)}')
print(f'con_eeg_file: \t\t{len(con_eeg_file)}')
print('#'*30)
print(f'check total files: \t{len(alz_eeg_file)+len(dem_eeg_file)+len(con_eeg_file)}')

## graph: plotting the signals 

In [ ]:
temp_data = mne.io.read_raw_eeglab(alz_eeg_file[0], preload=True) #read the data file.
temp_data.info #print out the information of the data.

In [ ]:
temp_data = mne.io.read_raw_eeglab(alz_eeg_file[0], preload=True) #read the data file.
print(temp_data)
#int_chnl = ['Fp1', 'Fp2', 'F1', 'F2', 'F3', 'F4', 'F7', 'F8', 'Fz']
int_chnl = None
temp_data, temp_time = temp_data.get_data(picks=int_chnl, tmin=None, tmax=None, return_times=True,)

fig = make_subplots(rows=temp_data.shape[0], cols=1, shared_xaxes=True, vertical_spacing=0.01, )
# Add traces to each subplot
for f in range(temp_data.shape[0]):
    fig.add_trace(go.Scatter(
        x=temp_time, 
        y=temp_data[f],
    ), row=f+1, col=1, )

# Update layout only once to set font sizes and remove margins
fig.update_layout(
    width=1700, 
    height=1500, 
    #xaxis=dict(tickfont=dict(size=20), showgrid=False, ),  # Set font size for x-axis ticks
    #yaxis=dict(tickfont=dict(size=20), showgrid=False, ),  # Set font size for y-axis ticks
    title='', 
    showlegend=False,
    margin=dict(l=10, r=10, t=10, b=10),  # Reduce margins around the plot
)

fig.show()

# Jensen-Shannon Distance

In [ ]:
# @title estimate PDF of each participant for each channels of the participants

def pdf_est(sig):
    assert isinstance(sig, np.ndarray), 'sig: given signal should be presented in numpy.array.'

    temp_bins = int(np.ceil(2 * (sig.shape[1])**(1/3)))
    temp_h, temp_edge = np.histogram(sig, bins=temp_bins, density=True, range=(-44, 44))
    temp_pmf = temp_h * np.diff(temp_edge)[0]

    return temp_h, temp_edge, temp_pmf

#list of the data file of the interested group; [alz_eeg_file] for alzheimer, [con_eeg_file] for healthy, [dem_eeg_file] for frontotemporal; please refer to the top.
int_eeg_file = dem_eeg_file 

#naming of the file, please change it as you do the calculation for different group; please use [alzheimer] for alzheimer, [control] for healthy, [dementia] for frontotemporal.
file_name = 'dementia' 
file_path = '/media/hengjie/4TB/entropy_combination_jensen/newdata2' #path of the file to be saved
int_chnl_all = [['C3', 'C4', 'Cz'], ['F3', 'F4', 'F7', 'F8', 'Fp1', 'Fp2', 'Fz'], ['O1', 'O2'], ['P3', 'P4', 'Pz'], ['T3', 'T4', 'T5', 'T6']] #interested channels
list_freq = [(0.5, 5), (5, 8), (8, 16), (16, 32), (32, 100)]

for int_chnl in int_chnl_all:
    for freq_index in range(len(list_freq)):
        fmin, fmax = np.min(list_freq[freq_index]), np.max(list_freq[freq_index])
        elec_data = ''.join(int_chnl)

        pdf_amp = []
        pdf_range = []
        pmf_amp = []



        for i in tqdm(int_eeg_file, position=0,): 
            temp_data = mne.io.read_raw_eeglab(i, preload=True) #read the data
            if (fmin > 0) and (fmax > 0):
                temp_data = temp_data.filter(fmin, fmax) #filtering the signal to specific frequency band. 
            temp_data = temp_data.pick(int_chnl)
            temp_h, temp_edge, temp_pmf = zip(*Parallel(n_jobs=-1)(delayed(pdf_est)(temp_data[i][0]) for i in range(temp_data[temp_data.ch_names][0].shape[0])))

            pdf_amp.append(temp_h)
            pdf_range.append(temp_edge)
            pmf_amp.append(temp_pmf)


        for l in tqdm(range(len(int_eeg_file)), position=0):
            participant_loc = l
            participant_pdf = pdf_amp[participant_loc]
            participant_range = pdf_range[participant_loc]
            participant_pmf = pmf_amp[participant_loc]

            js_participant = np.zeros((len(participant_pdf), len(participant_pdf)))
            for i in tqdm(range(len(participant_pdf)), position=0, leave=False,): 
                for j in range(len(participant_pdf)): 
                    temp_js = jensenshannon(participant_pmf[i], participant_pmf[j])
                    js_participant[i,j] = temp_js
            df_js = pd.DataFrame(js_participant)

            os.makedirs(f'{file_path}/{fmin}-{fmax}Hz_{elec_data}/{file_name}_js', exist_ok=True)
            df_js.to_csv(f'{file_path}/{fmin}-{fmax}Hz_{elec_data}/{file_name}_js/{file_name}_js_div_range100_sub{l+1}_freq{fmin}-{fmax}_chnl{elec_data}.tsv', index=False, header=temp_data.ch_names, sep='\t')


            plt.figure()
            plt.imshow(js_participant, cmap='Reds')
            plt.xticks(ticks=np.arange(0, len(temp_data.ch_names), 1), labels=temp_data.ch_names)
            plt.yticks(ticks=np.arange(0, len(temp_data.ch_names), 1), labels=temp_data.ch_names)
            plt.colorbar()
            plt.savefig(f'{file_path}/{fmin}-{fmax}Hz_{elec_data}/{file_name}_js/{file_name}_js_div_range100_sub{l+1}_freq{fmin}-{fmax}_chnl{elec_data}.png')
            plt.show()

os.system('spd-say "it is done. check the result."') # feature to make a sound; you are free to remove it. 

## combine all three

In [ ]:
# @title Extract the [Jensen divergence square matrix] from .tsv file. 
file_path = '/media/hengjie/4TB/entropy_combination_jensen/newdata2' #data path
int_chnl = 'F3', 'F4', 'F7', 'F8', 'Fp1', 'Fp2', 'Fz' #frontal; note the arrangement is very important. 
elec_data = ''.join(int_chnl) #user can copy and paste the channels name from the saved file instead. 
fmin, fmax = 32, 100 #frequency range: [0.5, 5], [5, 8], [8, 16], [16, 32], [32, 100]

###########################################################################
int_eeg_file1 = alz_eeg_file 
image_for1='alzheimer' 

add_matrix1 = []

for l in tqdm(range(len(int_eeg_file1)), position=0):
    file_name1 = f'{fmin}-{fmax}Hz_{elec_data}/{image_for1}_js/{image_for1}_js_div_range100_sub{l+1}_freq{fmin}-{fmax}_chnl{elec_data}.tsv'
    #file_name1 = f'{fmin}-{fmax}Hz/{image_for1}_js/{image_for1}_js_div_range100_sub{l+1}.tsv' # this is for no filter signals
    data_dir1 = os.path.join(file_path, file_name1)
    
    temp_df_data1 = pd.read_csv(data_dir1, sep='\t')
    
    add_matrix1.append(temp_df_data1)
    
add_matrix1 = np.array(add_matrix1)
sum_add_matrix1 = np.sum(add_matrix1, axis=0)
mean_add_matrix1 = np.mean(add_matrix1, axis=0)
std_add_matrix1 = np.std(add_matrix1, axis=0)
var_add_matrix1 = np.var(add_matrix1, axis=0)


###########################################################################
int_eeg_file2 = con_eeg_file 
image_for2='control'

add_matrix2 = []

for l in tqdm(range(len(int_eeg_file2)), position=0):
    file_name2 = f'{fmin}-{fmax}Hz_{elec_data}/{image_for2}_js/{image_for2}_js_div_range100_sub{l+1}_freq{fmin}-{fmax}_chnl{elec_data}.tsv'
    #file_name2 = f'{fmin}-{fmax}Hz/{image_for2}_js/{image_for2}_js_div_range100_sub{l+1}.tsv'
    data_dir2 = os.path.join(file_path, file_name2)
    
    temp_df_data2 = pd.read_csv(data_dir2, sep='\t')
    
    add_matrix2.append(temp_df_data2)
    
add_matrix2 = np.array(add_matrix2)
sum_add_matrix2 = np.sum(add_matrix2, axis=0)
mean_add_matrix2 = np.mean(add_matrix2, axis=0)
std_add_matrix2 = np.std(add_matrix2, axis=0)
var_add_matrix2 = np.var(add_matrix2, axis=0)

###########################################################################
int_eeg_file3 = dem_eeg_file 
image_for3='dementia'

add_matrix3 = []

for l in tqdm(range(len(int_eeg_file3)), position=0):
    file_name3 = f'{fmin}-{fmax}Hz_{elec_data}/{image_for3}_js/{image_for3}_js_div_range100_sub{l+1}_freq{fmin}-{fmax}_chnl{elec_data}.tsv'
    #file_name3 = f'{fmin}-{fmax}Hz/{image_for3}_js/{image_for3}_js_div_range100_sub{l+1}.tsv'
    data_dir3 = os.path.join(file_path, file_name3)
    
    temp_df_data3 = pd.read_csv(data_dir3, sep='\t')
    
    add_matrix3.append(temp_df_data3)
    
add_matrix3 = np.array(add_matrix3)
sum_add_matrix3 = np.sum(add_matrix3, axis=0)
mean_add_matrix3 = np.mean(add_matrix3, axis=0)
std_add_matrix3 = np.std(add_matrix3, axis=0)
var_add_matrix3 = np.var(add_matrix3, axis=0)

## statistical test

In [ ]:
from scipy.stats import kruskal

########################################################################
chnl_array1 = np.array((temp_df_data1.keys().tolist()))

chnl_matrix1 = [[row_i + '-' + row_j for row_j in chnl_array1] for row_i in chnl_array1]
chnl_matrix1 = np.array(chnl_matrix1)

uptri_ind1 = np.triu_indices(chnl_matrix1.shape[0], k=1)
chnl_comb1 = chnl_matrix1[uptri_ind1]
chnl_comb1 = np.array([s + f'_{image_for1}' for s in chnl_comb1])

boxplot_add_matrix1 = []
for i in tqdm(range(add_matrix1.shape[0]), position=0): 
    temp_matrix1 = add_matrix1[i][uptri_ind1]
    boxplot_add_matrix1.append(temp_matrix1)
    
boxplot_add_matrix1 = np.array(boxplot_add_matrix1)

########################################################################

chnl_array2 = np.array((temp_df_data2.keys().tolist()))

chnl_matrix2 = [[row_i + '-' + row_j for row_j in chnl_array2] for row_i in chnl_array2]
chnl_matrix2 = np.array(chnl_matrix2)

uptri_ind2 = np.triu_indices(chnl_matrix2.shape[0], k=1)
chnl_comb2 = chnl_matrix2[uptri_ind2]
chnl_comb2 = np.array([s + f'_{image_for2}' for s in chnl_comb2])

boxplot_add_matrix2 = []
for i in tqdm(range(add_matrix2.shape[0]), position=0): 
    temp_matrix2 = add_matrix2[i][uptri_ind2]
    boxplot_add_matrix2.append(temp_matrix2)
    
boxplot_add_matrix2 = np.array(boxplot_add_matrix2)

########################################################################

chnl_array3 = np.array((temp_df_data3.keys().tolist()))

chnl_matrix3 = [[row_i + '-' + row_j for row_j in chnl_array3] for row_i in chnl_array3]
chnl_matrix3 = np.array(chnl_matrix3)

uptri_ind3 = np.triu_indices(chnl_matrix3.shape[0], k=1)
chnl_comb3 = chnl_matrix3[uptri_ind3]
chnl_comb3 = np.array([s + f'_{image_for3}' for s in chnl_comb3])

boxplot_add_matrix3 = []
for i in tqdm(range(add_matrix3.shape[0]), position=0): 
    temp_matrix3 = add_matrix3[i][uptri_ind3]
    boxplot_add_matrix3.append(temp_matrix3)
    
boxplot_add_matrix3 = np.array(boxplot_add_matrix3)

########################################################################

#predefine empty set of the significant data
krus_test_results = []
krus_p_values = []
krus_sign_elec = []
krus_sign_index = []


for i in range(len(chnl_comb1)): 
    temp_krus_stat, temp_krus_pval = kruskal(boxplot_add_matrix1[:,i], boxplot_add_matrix2[:,i], boxplot_add_matrix3[:,i],)
    
    temp_alpha_pval = 0.05
    #collect the significant value of Kruskal-Wallis test
    if temp_krus_pval < temp_alpha_pval: 
        print("Reject the null hypothesis. There are significant differences between the groups.")
        print(f'H stat & p-val: {temp_krus_stat} & {temp_krus_pval}')
        print(f'electrode combination of {chnl_comb1[i]} at index {i}')
        krus_test_results.append(temp_krus_stat)
        krus_p_values.append(temp_krus_pval)
        krus_sign_elec.append(chnl_comb1[i])
        krus_sign_index.append(i)

htest_sig = np.vstack((krus_test_results, krus_p_values, krus_sign_elec, krus_sign_index))
htest_sig = htest_sig.T 
df_htest = pd.DataFrame(htest_sig, columns=['h_test', 'p_val', 'elec_comb', 'elec_comb_index'])
df_htest.to_csv(f'{file_path}/{fmin}-{fmax}Hz_{elec_data}/htest_js_div_range100_allsub_freq{fmin}-{fmax}_chnl{elec_data}.tsv', index=False, sep='\t')


print('DONE!!!')

## significant plot for Kruskal-Wallies test

In [ ]:
# @title significant plot for Kruskal-Wallies test
import plotly.graph_objects as go

fig = go.Figure()

for i in krus_sign_index:
    fig.add_trace(go.Box(
        y=boxplot_add_matrix1[:, i],
        name=f'{chnl_comb1[i]}',
        marker_color='blue', 
    ))
    fig.add_trace(go.Box(
        y=boxplot_add_matrix2[:, i],
        name=f'{chnl_comb2[i]}',
        marker_color='red'
    ))
    fig.add_trace(go.Box(
        y=boxplot_add_matrix3[:, i],
        name=f'{chnl_comb3[i]}',
        marker_color='green'
    ))

# Add statistical test values and p-values annotations
annotations = []
for i in range(len(krus_sign_index)):
    annotations.append(
        dict(
            x=(i * 3) + 1,  # Adjust x-coordinate for positioning
            y=min(max(boxplot_add_matrix1[:, i]), max(boxplot_add_matrix2[:, i]), max(boxplot_add_matrix3[:, i])) + 0.05,  # Adjust y-coordinate for positioning
            xref='x',
            yref='y',
            text=f"p = {krus_p_values[i]:.1f}",
            showarrow=False,
            font=dict(
                size=20,
                color="black"
            )
        )
    )
    annotations.append(
        dict(
            x=(i * 3) + 1,  # Adjust x-coordinate for positioning
            y=min(max(boxplot_add_matrix1[:, i]), max(boxplot_add_matrix2[:, i]), max(boxplot_add_matrix3[:, i])) + 0.03,  # Adjust y-coordinate for positioning
            xref='x',
            yref='y',
            text=f"H-test: {krus_test_results[i]:.1f}",
            showarrow=False,
            font=dict(
                size=20,
                color="black"
            )
        )
    )

fig.update_layout(
    annotations=annotations,
    #width=3500,
    #height=600,
    showlegend=False,
    xaxis=dict(
        title='Group',
        titlefont=dict(size=20),  # X-axis title font size
        tickfont=dict(size=25)    # X-axis tick labels font size
    ),
    yaxis=dict(
        title='Jensen-Shannon Divergence',
        titlefont=dict(size=30),  # Y-axis title font size
        tickfont=dict(size=25)   # Y-axis tick labels font size
    ),
)

#fig.update_yaxes(range=[0.002, 0.5])
output_dir = f'{file_path}/figures/{fmin}-{fmax}Hz_{elec_data}'
os.makedirs(output_dir, exist_ok=True)
pio.write_html(fig, f'{output_dir}/htest_js_div_range100_boxplot_freq{fmin}-{fmax}_chnl{elec_data}.html', include_plotlyjs='cdn')

os.makedirs(f'{file_path}/{fmin}-{fmax}Hz', exist_ok=True)
pio.write_html(fig, f'{file_path}/{fmin}-{fmax}Hz/htest_js_div_range100_boxplot_freq{fmin}-{fmax}.html', include_plotlyjs='cdn')

os.makedirs(f'{file_path}/{fmin}-{fmax}Hz_{elec_data}', exist_ok=True)
pio.write_html(fig, f'{file_path}/{fmin}-{fmax}Hz_{elec_data}/htest_js_div_range100_violinplot_freq{fmin}-{fmax}_chnl{elec_data}.html', include_plotlyjs='cdn')
fig.show()

# information rate calculation
The following code require the code from the top to get the file paths for all the participants. 

In [ ]:
int_eeg_file = alz_eeg_file + con_eeg_file + dem_eeg_file #list of file path of all the data
file_path = '/media/hengjie/4TB/info_rate/newdata2' #file path for data to be save; IMPORTANT change the file path for your own use. 

int_chnl_all = [['C3', 'C4', 'Cz'], ['F3', 'F4', 'F7', 'F8', 'Fp1', 'Fp2', 'Fz'], ['O1', 'O2'], ['P3', 'P4', 'Pz'], ['T3', 'T4', 'T5', 'T6']] #interested channels
list_freq = [(0.5, 5), (5, 8), (8, 16), (16, 32), (32, 100)]

for int_chnl in int_chnl_all:
    for freq_index in range(len(list_freq)):
        # frequency of the signals; signals will not be filtered if put fmin, fmax = 0, 0 (check the code within)
        fmin, fmax = np.min(list_freq[freq_index]), np.max(list_freq[freq_index])

        int_second_win = ((1/fmin) + (1/fmax))*0.5 #time in term of second for the sampling window size; it is based on the mid-point of lower and upper bound of interested frequency band. 
        int_second_sld = 0.5*int_second_win #percentage the sliding window with respect to the sampling window

        # Channels to select
        #int_chnl = 'O1', 'O2' #occipital
        chnl_name = ''.join(int_chnl)

        for f in range(len(int_eeg_file)): 
            # read the file of the signals
            temp_file_name = int_eeg_file[f]
            temp_data = mne.io.read_raw_eeglab(temp_file_name, preload=True)
            temp_participant_id = temp_file_name.split('/')[-3]

            # filter the signals; signals will not be filtered if put fmin, fmax = 0,0; 
            # user can remove it as same result would be expected if fmin, fmax = None, None
            if (fmin > 0) and (fmax > 0): 
                temp_data = temp_data.filter(fmin, fmax)


            # get all the signals in array. 
            temp_data = temp_data.pick(int_chnl)
            temp_data_all = temp_data.get_data() #extract all data of the signals
            temp_data_time = temp_data.times #extract the time data
            temp_data_freq = temp_data.info['sfreq'] #sampling frequency of the data
            print(f'data sampling frequency: {temp_data_freq}')

            # sampling window and sliding window data; shows how many data points are included in the window
            temp_sample_win = int(int_second_win * temp_data_freq)
            temp_slide_win = int(int_second_sld * temp_data_freq)
            print(f'sampling data: \t {temp_sample_win}')
            print(f'sliding data: \t {temp_slide_win}')

            temp_bins_size = int(np.ceil(2 * (temp_sample_win*temp_data_all.shape[0])**(1/3))) # bin size for the information rate based on [Rice rule]
            print(f'rice rule bins: \t {temp_bins_size}')

            # sliding window of the data
            temp_data_infoall = sliding_window_view(temp_data_all, window_shape=temp_sample_win, axis=-1)
            temp_data_infoall = temp_data_infoall[:, ::temp_slide_win, :]

            # sliding window of the time
            temp_data_infotime = sliding_window_view(temp_data_time, window_shape=temp_sample_win, axis=-1)
            temp_data_infotime = temp_data_infotime[::temp_slide_win, :]

            # information rate calculation; joblib parallalization is used
            temp_inforate_data, temp_inforate_time = zip(*Parallel(n_jobs=-1)(delayed(ig.adj_collect_inforate_square)(data=temp_data_infoall, time=temp_data_infotime, i=i, bins_size=temp_bins_size) for i in range(temp_data_infotime.shape[0]-1)))
            df_info = np.vstack((temp_inforate_time, temp_inforate_data)).T
            df_info = pd.DataFrame(df_info, columns=['time', 'info_rate_square'])


            # saving the calculated data in interested folder
            output_dir = f'{file_path}/{fmin}-{fmax}Hz_{chnl_name}_sampfreq{temp_data_freq}_winsecond{int_second_win}_sldsecond{int_second_sld}'
            os.makedirs(output_dir, exist_ok=True) # create the folder if the folder is not exist.
            # naming the file for the calculated data; IMPORTANT change the file naming after {output_dir}/
            output_file = f'{output_dir}/inforate_freq{fmin}-{fmax}_{chnl_name}_sampfreq{temp_data_freq}_winsecond{int_second_win}_sldsecond{int_second_sld}_{temp_participant_id}'
            df_info.to_csv(f'{output_file}.tsv', sep='\t', index=False)
    
os.system('spd-say "it is done. check the result."') # feature to make a sound; you are free to remove it. 

## graph: PDF for the information rate over time for each group with standard deviation

In [ ]:
# @title PDF for the information rate over time for each group with standard deviation

folder_dir = '/media/hengjie/4TB/info_rate/newdata2' #directory of the data
save_folder_dir = '/media/hengjie/4TB/info_rate/newdata2/figures' #directory of the saved images

fmin, fmax = 16, 32 # frequency of the signals
sampfreq = 500.0 # sampling frequency of the signals
int_win = ((1/fmin) + (1/fmax))*0.5 #time in term of second for the sampling window size; it is based on the mid-point of lower and upper bound of interested frequency band. 
int_sld = int_win*0.5
print(int_win, int_sld)

'''
range of the distribution; NOTE: please set to appropriate range; 
the suggested ranges are [0 to 3.5] for [0.5 to 5Hz], [0 to 25] for [5 to 8Hz], [1 to 37] for [8 to 16Hz], [5 to 75] for [16 to 32Hz], and [5 to 160] for [32 to 100Hz]. 
'''
int_range = [0, 74] 

int_bins = 40 # bin size of the distribution
int_chnl = 'C3C4Cz' # channels of the signals
file_dir = f'{fmin}-{fmax}Hz_{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}'

list_sub = participants_df['participant_id']

common_alz_group = []
common_con_group = []
common_dem_group = []

alz_shannon_entropy = []
con_shannon_entropy = []
dem_shannon_entropy = []


for l in range(len(list_sub)):
    int_sub = list_sub[l]
    temp_info_data = pd.read_csv(f'{folder_dir}/{file_dir}/inforate_freq{fmin}-{fmax}_{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}_{int_sub}.tsv', sep='\t')
    temp_pdf, temp_range = np.histogram(temp_info_data['info_rate_square']**0.5, bins=int_bins, density=True, range=int_range,)
    temp_range = (temp_range[1:]+temp_range[:-1])/2
    temp_pmf = temp_pdf * np.diff(temp_range)[0]
    
    common_alz = np.intersect1d(int_sub, alz_part['participant_id'].to_numpy())
    common_con = np.intersect1d(int_sub, con_part['participant_id'].to_numpy())
    common_dem = np.intersect1d(int_sub, dem_part['participant_id'].to_numpy())
    
    if common_alz.shape[0] == 1:
        common_alz_group.append(temp_pmf)

    if common_con.shape[0] == 1:
        common_con_group.append(temp_pmf)
    
    if common_dem.shape[0] == 1:
        common_dem_group.append(temp_pmf)

        
common_alz_group = np.array(common_alz_group)
common_con_group = np.array(common_con_group)
common_dem_group = np.array(common_dem_group)

common_alz_group_mean = np.mean(common_alz_group, axis=0)
common_con_group_mean = np.mean(common_con_group, axis=0)
common_dem_group_mean = np.mean(common_dem_group, axis=0)

common_alz_group_std = np.std(common_alz_group, axis=0)
common_con_group_std = np.std(common_con_group, axis=0)
common_dem_group_std = np.std(common_dem_group, axis=0)

common_alz_group_upper = common_alz_group_mean + common_alz_group_std
common_alz_group_lower = common_alz_group_mean - common_alz_group_std

common_con_group_upper = common_con_group_mean + common_con_group_std 
common_con_group_lower = common_con_group_mean - common_con_group_std

common_dem_group_upper = common_dem_group_mean + common_dem_group_std 
common_dem_group_lower = common_dem_group_mean - common_dem_group_std

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=np.concatenate([temp_range, temp_range[::-1]]),  # x followed by x reversed
    y=np.concatenate([common_alz_group_upper, common_alz_group_lower[::-1]]),  # upper bound followed by lower bound reversed
    fill='toself',
    fillcolor='rgba(0, 0, 255, 0.2)',  # semi-transparent fill color
    line=dict(color='rgba(0, 0, 255, 0.2)'),
    hoverinfo="skip",
    showlegend=False,
    name='Alzheimer_error',
))
fig.add_trace(go.Scatter(
    x=temp_range, 
    y=common_alz_group_mean, 
    name='Alzheimer',
    line=dict(color='rgba(0, 0, 255, 1)'),
))


fig.add_trace(go.Scatter(
    x=np.concatenate([temp_range, temp_range[::-1]]),  # x followed by x reversed
    y=np.concatenate([common_con_group_upper, common_con_group_lower[::-1]]),  # upper bound followed by lower bound reversed
    fill='toself',
    fillcolor='rgba(255, 0, 0, 0.2)',  # semi-transparent fill color
    line=dict(color='rgba(255, 0, 0, 0.2)'),
    hoverinfo="skip",
    showlegend=False,
    name='control_error',
))
fig.add_trace(go.Scatter(
    x=temp_range, 
    y=common_con_group_mean, 
    name='control',
    line=dict(color='rgba(255, 0, 0, 1)'),
))


fig.add_trace(go.Scatter(
    x=np.concatenate([temp_range, temp_range[::-1]]),  # x followed by x reversed
    y=np.concatenate([common_dem_group_upper, common_dem_group_lower[::-1]]),  # upper bound followed by lower bound reversed
    fill='toself',
    fillcolor='rgba(0, 255, 0, 0.2)',  # semi-transparent fill color
    line=dict(color='rgba(0, 255, 0, 0.2)'),
    hoverinfo="skip",
    showlegend=False,
    name='frontotemporal_error',
))
fig.add_trace(go.Scatter(
    x=temp_range, 
    y=common_dem_group_mean, 
    name='frontotemporal',
    line=dict(color='rgba(0, 255, 0, 1)'),
))



fig.update_layout(
    width=600, 
    height=500, 
    yaxis=dict(
        #title=dict(text='PMF', font=dict(size=45), ), 
        tickfont=dict(size=30), 
        #showgrid=False, 
        #type='log',
    ),
    xaxis=dict(
        #title=dict(text='information rate', font=dict(size=45), ), 
        tickfont=dict(size=30), 
        #showgrid=False, 
        #type='log',
    ), 
    legend=dict(
        font=dict(size=20), 
    ), 
    #title=f'{int_chnl}_win{int_win}_sld{int_sld}_freq{fmin}-{fmax}_bins{int_bins}'
    title='', 
    showlegend=False, 
    margin=dict(l=10, r=10, t=10, b=10, ),
)

os.makedirs(save_folder_dir, exist_ok=True) # create the folder if the folder is not exist.
pio.write_html(fig, f'{save_folder_dir}/info_rate_pmf_freq{fmin}-{fmax}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}_bins{int_bins}_{int_chnl}.html', include_plotlyjs='cdn')
fig.show()

## graph & calculation: Shannon Entropy information rate - for PDF of information rate over time for each group 

In [ ]:
# graph & calculation: Shannon Entropy information rate - for PDF of information rate over time for each group 
folder_dir = '/media/hengjie/4TB/info_rate/newdata2' #folder where the data is stored

list_freqrange = [(0.5, 5.0), (5, 8), (8, 16), (16, 32), (32, 100)]
list_winsize = [1.1, 0.1625, 0.09375, 0.046875, 0.020625]
print(len(list_freqrange) == len(list_winsize)) #for checking the [frequency band] and the [window size] list having same length

df_shainfo_meanstd = []
df_shainfo_medianq1q3 = []

kruskal_test_data = []
dunn_test_data = {}


for r in range(len(list_freqrange)):
    '''
    PLEASE CHECK this section before running the code especially the [int_chnl]
    '''
    fmin, fmax = list_freqrange[r]
    sampfreq = 500.0 #sampling frequency
    int_win = list_winsize[r]
    int_sld = int_win*0.5
    print(int_win, int_sld)
    int_chnl = 'C3C4Cz'  #interested channel for the analysis
    file_dir = f'{fmin}-{fmax}Hz_{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}'

    list_sub = participants_df['participant_id']

    common_alz_group = []
    common_con_group = []
    common_dem_group = []

    for l in tqdm(range(len(list_sub)), position=0):
        int_sub = list_sub[l]
        temp_info_data = pd.read_csv(f'{folder_dir}/{file_dir}/inforate_freq{fmin}-{fmax}_{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}_{int_sub}.tsv', sep='\t')
        #print(temp_info_data.shape)
        rice_bins = int(np.ceil(2 * (temp_info_data.shape[0])**(1/3))) #rice rule bin size. 
        #print(f'rice bins: {rice_bins}')
        temp_pdf, temp_range = np.histogram(temp_info_data['info_rate_square']**0.5, bins=rice_bins, density=True,)
        temp_range = (temp_range[1:]+temp_range[:-1])/2
        temp_pmf = temp_pdf * np.diff(temp_range)[0]

        temp_shannon_inforate = entropy(temp_pmf)#/entropy(np.ones(temp_pdf.shape[0])/(temp_pdf.shape[0]))

        common_alz = np.intersect1d(int_sub, alz_part['participant_id'].to_numpy())
        common_con = np.intersect1d(int_sub, con_part['participant_id'].to_numpy())
        common_dem = np.intersect1d(int_sub, dem_part['participant_id'].to_numpy())

        if common_alz.shape[0] == 1:
            common_alz_group.append(temp_shannon_inforate)

        if common_con.shape[0] == 1:
            common_con_group.append(temp_shannon_inforate)

        if common_dem.shape[0] == 1:
            common_dem_group.append(temp_shannon_inforate)

    #mean and standard deviation        
    common_alz_group = np.array(common_alz_group)
    common_con_group = np.array(common_con_group)
    common_dem_group = np.array(common_dem_group)

    common_alz_group_mean = np.mean(common_alz_group, axis=0)
    common_con_group_mean = np.mean(common_con_group, axis=0)
    common_dem_group_mean = np.mean(common_dem_group, axis=0)

    common_alz_group_std = np.std(common_alz_group, axis=0)
    common_con_group_std = np.std(common_con_group, axis=0)
    common_dem_group_std = np.std(common_dem_group, axis=0)
    

    #median and quantiles
    common_alz_group_median = np.median(common_alz_group)
    common_con_group_median = np.median(common_con_group)
    common_dem_group_median = np.median(common_dem_group)

    common_alz_group_q1 = np.quantile(common_alz_group, 0.25)
    common_con_group_q1 = np.quantile(common_con_group, 0.25)
    common_dem_group_q1 = np.quantile(common_dem_group, 0.25)

    common_alz_group_q3 = np.quantile(common_alz_group, 0.75)
    common_con_group_q3 = np.quantile(common_con_group, 0.75)
    common_dem_group_q3 = np.quantile(common_dem_group, 0.75)


    temp_df_shainfo_meanstd = np.vstack(( f'{np.round(common_alz_group_mean, 4)}/{np.round(common_alz_group_std, 4)}', 
                                    f'{np.round(common_con_group_mean, 4)}/{np.round(common_con_group_std, 4)}', 
                                    f'{np.round(common_dem_group_mean, 4)}/{np.round(common_dem_group_std, 4)}' ))
    temp_df_shainfo_medianq1q3 = np.vstack(( f'{np.round(common_alz_group_median, 4)}/{np.round(common_alz_group_q1, 4)}/{np.round(common_alz_group_q3, 4)}', 
                                       f'{np.round(common_con_group_median, 4)}/{np.round(common_con_group_q1, 4)}/{np.round(common_con_group_q3,4)}', 
                                       f'{np.round(common_dem_group_median, 4)}/{np.round(common_dem_group_q1, 4)}/{np.round(common_dem_group_q3, 4)}' ))
    
    #append save the data
    df_shainfo_meanstd.append(temp_df_shainfo_meanstd)
    df_shainfo_medianq1q3.append(temp_df_shainfo_medianq1q3)


    ###Statistical Test###
    kruskal_test, kruskal_pval = kruskal(common_alz_group, common_con_group, common_dem_group)
    print(f'kruskal: {kruskal_test}, {kruskal_pval}')
    
    #append save the data
    kruskal_test_data.append(f'{np.round(kruskal_test, 4)}/{np.round(kruskal_pval, 4)}')
    
    #Dunn test (after kruskal)
    temp_p_val_dunn = sp.posthoc_dunn([common_alz_group, common_con_group, common_dem_group], p_adjust='holm')
    temp_p_val_dunn.index = ['alz', 'con', 'dem'] #label the rows
    temp_p_val_dunn.columns = ['alz', 'con', 'dem'] #label the columns
    
    dunn_test_data.update({f'{list_freqrange[r][0]}-{list_freqrange[r][1]}': temp_p_val_dunn}) #update the dictionary with the combination as the key
    
    
    
    # plot for the first time
    fig = go.Figure()
    fig.add_trace(go.Box(
        y=common_alz_group, 
        boxmean=True, 
        name='AD',
    ))
    fig.add_trace(go.Box(
        y=common_con_group,
        boxmean=True, 
        name='HC',
    ))
    fig.add_trace(go.Box(
        y=common_dem_group, 
        boxmean=True, 
        name='FT',
    ))
    fig.update_layout(
        width=600, 
        height=500, 
        xaxis=dict(tickfont=dict(size=35), title='', showgrid=False,), 
        yaxis=dict(tickfont=dict(size=30), title='', showgrid=False, ),
        #title=f'shannon_inforate_{fmin}-{fmax}Hz_comb{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}',
        title = '', 
        showlegend=False,
        margin=dict(l=10, r=10, t=10, b=10, ),
    )
    os.makedirs(f'{folder_dir}/shannon_inforate', exist_ok=True) # create the folder if the folder is not exist.
    pio.write_image(fig, f'{folder_dir}/shannon_inforate/shannon_inforate_{int_chnl}_{list_freqrange[r][0]}-{list_freqrange[r][1]}Hz_sampfreq{sampfreq}_winsecond{list_winsize[r]}.png')
    fig.show()


    # plot second time to include the significant difference
    if kruskal_pval < 0.05:
        fig = go.Figure()
        fig.add_trace(go.Box(
            y=common_alz_group, 
            boxmean=True, 
            name='AD',
        ))
        fig.add_trace(go.Box(
            y=common_con_group,
            boxmean=True, 
            name='HC',
        ))
        fig.add_trace(go.Box(
            y=common_dem_group, 
            boxmean=True, 
            name='FT',
        ))
        fig.update_layout(
            width=600, 
            height=500, 
            xaxis=dict(tickfont=dict(size=35), title='', showgrid=False,), 
            yaxis=dict(tickfont=dict(size=30), title='', showgrid=False, ),
            #title=f'shannon_inforate_{fmin}-{fmax}Hz_comb{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}',
            title = '', 
            showlegend=False,
            margin=dict(l=10, r=10, t=10, b=10, ),
        )
        xcoor_group = np.where(dunn_test_data[f'{fmin}-{fmax}']<0.05)[0]
        ycoor_group = np.where(dunn_test_data[f'{fmin}-{fmax}']<0.05)[1]
        print(xcoor_group, ycoor_group)
        for xc in xcoor_group:
            for yc in ycoor_group:
                # Determine the maximum y coordinate for the line
                if (xc, yc) == (0, 1):
                    y_line = max(np.quantile(common_alz_group, 0.75), np.quantile(common_con_group, 0.75))
                elif (xc, yc) == (0, 2):
                    y_line = max(np.quantile(common_alz_group, 0.75), np.quantile(common_dem_group, 0.75))
                elif (xc, yc) == (1, 2):
                    y_line = max(np.quantile(common_con_group, 0.75), np.quantile(common_dem_group, 0.75))
                else:
                    continue  # Skip if the condition is not met

                # Draw the line
                fig.add_shape(
                    type='line', 
                    x0=xc, x1=yc, 
                    y0=y_line, y1=y_line,
                    line=dict(color='black', width=1)
                )

                # Add a significance bracket (example between Group 1 and Group 2)
                fig.add_annotation(
                    x=0.5*(xc+yc),  # Midpoint between Group 1 and Group 2
                    y=y_line,  # Y position above the boxes
                    text="* p < 0.05",  # Significance label
                    showarrow=False,
                    font=dict(size=30),
                )
        os.makedirs(f'{folder_dir}/shannon_inforate', exist_ok=True) # create the folder if the folder is not exist.
        pio.write_image(fig, f'{folder_dir}/shannon_inforate/shannon_inforate_{int_chnl}_{list_freqrange[r][0]}-{list_freqrange[r][1]}Hz_sampfreq{sampfreq}_winsecond{list_winsize[r]}.png')
        fig.show()


    
#saving the data in .tsv format.
df_shainfo_meanstd_data = pd.DataFrame(np.squeeze(np.array(df_shainfo_meanstd)).T, index=['alz', 'con', 'dem'], columns=[f'{j[0]}-{j[1]}' for j in list_freqrange])
df_shainfo_medianq1q3_data = pd.DataFrame(np.squeeze(np.array(df_shainfo_medianq1q3)).T, index=['alz', 'con', 'dem'], columns=[f'{j[0]}-{j[1]}' for j in list_freqrange])

os.makedirs(f'{folder_dir}/shannon_inforate/shannon_inforate_meanstd', exist_ok=True) # create the folder if the folder is not exist.
os.makedirs(f'{folder_dir}/shannon_inforate/shannon_inforate_medianq1q3', exist_ok=True) # create the folder if the folder is not exist.
df_shainfo_meanstd_data.to_csv(f'{folder_dir}/shannon_inforate/shannon_inforate_meanstd/shannon_inforate_meanstd_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}.tsv', sep='\t')
df_shainfo_medianq1q3_data.to_csv(f'{folder_dir}/shannon_inforate/shannon_inforate_medianq1q3/shannon_inforate_medianq1q3_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}.tsv', sep='\t')

os.makedirs(f'{folder_dir}/shannon_inforate/kruskal_shannon_inforate', exist_ok=True) # create the folder if the folder is not exist.
kruskal_test_data = pd.DataFrame(np.expand_dims(np.array(kruskal_test_data), axis=0), index=[f'{int_chnl}'], columns=[f'{j[0]}-{j[1]}' for j in list_freqrange])
kruskal_test_data.to_csv(f'{folder_dir}/shannon_inforate/kruskal_shannon_inforate/kruskal_shannon_inforate_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}.tsv', sep='\t')

np.save(f'{folder_dir}/shannon_inforate/kruskal_shannon_inforate/dunn_shannon_inforate_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}',  dunn_test_data) #save the dictionary data in .npy file


## graph & calculation: Moment of the PDF of inforamtion rate over time for each group

In [ ]:
# graph & calculation: Moment of the PDF of inforamtion rate over time for each group

from scipy.stats import moment

folder_dir = '/media/hengjie/4TB/info_rate/newdata2' #folder where the data is stored. 

list_freqrange = [(0.5, 5.0), (5, 8), (8, 16), (16, 32), (32, 100)]
list_winsize = [1.1, 0.1625, 0.09375, 0.046875, 0.020625]
print(len(list_freqrange) == len(list_winsize)) #for checking the [frequency band] and the [window size] list having same length

momentinfo_alz_data = []
momentinfo_con_data = []
momentinfo_dem_data = []

anova_test_moment1_data = []
anova_test_moment2_data = []
anova_test_moment3_data = []
anova_test_moment4_data = []

kruskal_test_moment1_data = []
kruskal_test_moment2_data = []
kruskal_test_moment3_data = []
kruskal_test_moment4_data = []

dunn_test_moment1_data = {}
dunn_test_moment2_data = {}
dunn_test_moment3_data = {}
dunn_test_moment4_data = {}

tukey_test_moment1_data = {}
tukey_test_moment2_data = {}
tukey_test_moment3_data = {}
tukey_test_moment4_data = {}

pairttest_test_moment1_data = {}
pairttest_test_moment2_data = {}
pairttest_test_moment3_data = {}
pairttest_test_moment4_data = {}

for r in range(len(list_freqrange)):
    '''
    PLEASE CHECK this section before running the code especially the [int_chnl]
    '''
    fmin, fmax = list_freqrange[r]
    sampfreq = 500.0 #sampling frequency
    int_win = list_winsize[r]
    int_sld = int_win*0.5
    print(int_win, int_sld)
    int_chnl = 'C3C4Cz' #interested channel for the analysis
    file_dir = f'{fmin}-{fmax}Hz_{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}'

    list_sub = participants_df['participant_id']

    alz_group_mean = []
    con_group_mean = []
    dem_group_mean = [] 
    
    alz_group_var = []
    con_group_var = []
    dem_group_var = []
    
    alz_group_skew = []
    con_group_skew = []
    dem_group_skew = []
    
    alz_group_kur = []
    con_group_kur = []
    dem_group_kur = []

    for l in tqdm(range(len(list_sub)), position=0):
        int_sub = list_sub[l]
        temp_info_data = pd.read_csv(f'{folder_dir}/{file_dir}/inforate_freq{fmin}-{fmax}_{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}_{int_sub}.tsv', sep='\t')
        #print(temp_info_data.shape)

        #central moment of calculation
        temp_mean = np.mean(temp_info_data['info_rate_square']**0.5)
        temp_moment2 = moment(temp_info_data['info_rate_square']**0.5, moment=2) #variance
        temp_moment3 = moment(temp_info_data['info_rate_square']**0.5, moment=3)
        temp_moment4 = moment(temp_info_data['info_rate_square']**0.5, moment=4)
        
        temp_norm_moment3 = temp_moment3 / (temp_moment2**(3/2)) #normalized skewness
        temp_norm_moment4 = (temp_moment4 / (temp_moment2**2)) - 3 #normalized kurtosis (excess kurtosis)
        

        common_alz = np.intersect1d(int_sub, alz_part['participant_id'].to_numpy())
        common_con = np.intersect1d(int_sub, con_part['participant_id'].to_numpy())
        common_dem = np.intersect1d(int_sub, dem_part['participant_id'].to_numpy())

        if common_alz.shape[0] == 1:
            alz_group_mean.append(temp_mean)
            alz_group_var.append(temp_moment2)
            alz_group_skew.append(temp_norm_moment3)
            alz_group_kur.append(temp_norm_moment4)

        if common_con.shape[0] == 1:
            con_group_mean.append(temp_mean)
            con_group_var.append(temp_moment2)
            con_group_skew.append(temp_norm_moment3)
            con_group_kur.append(temp_norm_moment4)

        if common_dem.shape[0] == 1:
            dem_group_mean.append(temp_mean)
            dem_group_var.append(temp_moment2)
            dem_group_skew.append(temp_norm_moment3)
            dem_group_kur.append(temp_norm_moment4)

    #1, 2, 3, and 4 moments
    alz_group_mean = np.array(alz_group_mean)
    alz_group_var = np.array(alz_group_var)
    alz_group_skew = np.array(alz_group_skew)
    alz_group_kur = np.array(alz_group_kur)
    
    con_group_mean = np.array(con_group_mean)
    con_group_var = np.array(con_group_var)
    con_group_skew = np.array(con_group_skew)
    con_group_kur = np.array(con_group_kur)
    
    dem_group_mean = np.array(dem_group_mean)
    dem_group_var = np.array(dem_group_var)
    dem_group_skew = np.array(dem_group_skew)
    dem_group_kur = np.array(dem_group_kur)
    
    
    temp_df_momentinfo_alz = np.vstack(( f'{alz_group_mean}', f'{alz_group_var}', 
                                        f'{alz_group_skew}', f'{alz_group_kur}' ))
    
    temp_df_momentinfo_con = np.vstack(( f'{con_group_mean}', f'{con_group_var}', 
                                        f'{con_group_skew}', f'{con_group_kur}' ))
    
    temp_df_momentinfo_dem = np.vstack(( f'{dem_group_mean}', f'{dem_group_var}', 
                                        f'{dem_group_skew}', f'{dem_group_kur}' ))
    
    
    #append save the data
    momentinfo_alz_data.append(temp_df_momentinfo_alz)
    momentinfo_con_data.append(temp_df_momentinfo_con)
    momentinfo_dem_data.append(temp_df_momentinfo_dem)


    ###Statistical Test###    
    kruskal_test_moment1, kruskal_pval_moment1 = kruskal(alz_group_mean, con_group_mean)#, dem_group_mean)
    kruskal_test_moment2, kruskal_pval_moment2 = kruskal(alz_group_var, con_group_var)#, dem_group_var)
    kruskal_test_moment3, kruskal_pval_moment3 = kruskal(alz_group_skew, con_group_skew)#, dem_group_skew)
    kruskal_test_moment4, kruskal_pval_moment4 = kruskal(alz_group_kur, con_group_kur)#, dem_group_kur)
    print(f'kruskal moment1: \t {kruskal_test_moment1}, {kruskal_pval_moment1}')
    print(f'kruskal moment2: \t {kruskal_test_moment2}, {kruskal_pval_moment2}')
    print(f'kruskal moment3: \t {kruskal_test_moment3}, {kruskal_pval_moment3}')
    print(f'kruskal moment4: \t {kruskal_test_moment4}, {kruskal_pval_moment4}')
    
    #append save the data
    kruskal_test_moment1_data.append(f'{np.round(kruskal_test_moment1, 4)}/{np.round(kruskal_pval_moment1, 4)}')
    kruskal_test_moment2_data.append(f'{np.round(kruskal_test_moment2, 4)}/{np.round(kruskal_pval_moment2, 4)}')
    kruskal_test_moment3_data.append(f'{np.round(kruskal_test_moment3, 4)}/{np.round(kruskal_pval_moment3, 4)}')
    kruskal_test_moment4_data.append(f'{np.round(kruskal_test_moment4, 4)}/{np.round(kruskal_pval_moment4, 4)}')
    
    
    
    #Dunn test (after kruskal)
    temp_p_val_dunn_mean = sp.posthoc_dunn([alz_group_mean, con_group_mean, dem_group_mean], p_adjust='holm')
    temp_p_val_dunn_mean.index = ['alz', 'con', 'dem'] #label the rows
    temp_p_val_dunn_mean.columns = ['alz', 'con', 'dem'] #label the columns
    
    dunn_test_moment1_data.update({f'{list_freqrange[r][0]}-{list_freqrange[r][1]}': temp_p_val_dunn_mean}) #update the dictionary with the combination as the key
    #############################
    
    temp_p_val_dunn_var = sp.posthoc_dunn([alz_group_var, con_group_var, dem_group_var], p_adjust='holm')
    temp_p_val_dunn_var.index = ['alz', 'con', 'dem'] #label the rows
    temp_p_val_dunn_var.columns = ['alz', 'con', 'dem'] #label the columns
    
    dunn_test_moment2_data.update({f'{list_freqrange[r][0]}-{list_freqrange[r][1]}': temp_p_val_dunn_var}) #update the dictionary with the combination as the key
    #############################
    
    temp_p_val_dunn_skew = sp.posthoc_dunn([alz_group_skew, con_group_skew, dem_group_skew], p_adjust='holm')
    temp_p_val_dunn_skew.index = ['alz', 'con', 'dem'] #label the rows
    temp_p_val_dunn_skew.columns = ['alz', 'con', 'dem'] #label the columns
    
    dunn_test_moment3_data.update({f'{list_freqrange[r][0]}-{list_freqrange[r][1]}': temp_p_val_dunn_skew}) #update the dictionary with the combination as the key
    #############################
    
    temp_p_val_dunn_kur = sp.posthoc_dunn([alz_group_kur, con_group_kur, dem_group_kur], p_adjust='holm')
    temp_p_val_dunn_kur.index = ['alz', 'con', 'dem'] #label the rows
    temp_p_val_dunn_kur.columns = ['alz', 'con', 'dem'] #label the columns
    
    dunn_test_moment4_data.update({f'{list_freqrange[r][0]}-{list_freqrange[r][1]}': temp_p_val_dunn_kur}) #update the dictionary with the combination as the key
    #############################
    
    
    # plot for the first time
    fig = go.Figure()
    fig.add_trace(go.Box(
        y=alz_group_mean, 
        boxmean=True, 
        name='AD',
    ))
    fig.add_trace(go.Box(
        y=con_group_mean,
        boxmean=True, 
        name='HC',
    ))
    fig.add_trace(go.Box(
        y=dem_group_mean, 
        boxmean=True, 
        name='FT',
    ))
    fig.update_layout(
        width=600, 
        height=500, 
        xaxis=dict(tickfont=dict(size=35), title='', showgrid=False,), 
        yaxis=dict(tickfont=dict(size=30), title='', showgrid=False, ),
        #title=f'mean_inforate_{fmin}-{fmax}Hz_comb{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}',
        title = '', 
        showlegend=False,
        margin=dict(l=10, r=10, t=10, b=10, ),
    )
    pio.write_image(fig, f'{folder_dir}/shannon_inforate/mean_inforate_{int_chnl}_{list_freqrange[r][0]}-{list_freqrange[r][1]}Hz_sampfreq{sampfreq}_winsecond{list_winsize[r]}.png')
    fig.show()
    
    
    
    fig = go.Figure()
    fig.add_trace(go.Box(
        y=alz_group_var, 
        boxmean=True, 
        name='AD',
    ))
    fig.add_trace(go.Box(
        y=con_group_var,
        boxmean=True, 
        name='HC',
    ))
    fig.add_trace(go.Box(
        y=dem_group_var, 
        boxmean=True, 
        name='FT',
    ))
    fig.update_layout(
        width=600, 
        height=500, 
        xaxis=dict(tickfont=dict(size=35), title='', showgrid=False,), 
        yaxis=dict(tickfont=dict(size=30), title='', showgrid=False, ),
        #title=f'var_inforate_{fmin}-{fmax}Hz_comb{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}',
        title = '', 
        showlegend=False,
        margin=dict(l=10, r=10, t=10, b=10, ),
    )
    pio.write_image(fig, f'{folder_dir}/shannon_inforate/var_inforate_{int_chnl}_{list_freqrange[r][0]}-{list_freqrange[r][1]}Hz_sampfreq{sampfreq}_winsecond{list_winsize[r]}.png')
    fig.show()
    
    
    fig = go.Figure()
    fig.add_trace(go.Box(
        y=alz_group_skew, 
        boxmean=True, 
        name='AD',
    ))
    fig.add_trace(go.Box(
        y=con_group_skew,
        boxmean=True, 
        name='HC',
    ))
    fig.add_trace(go.Box(
        y=dem_group_skew, 
        boxmean=True, 
        name='FT',
    ))
    fig.update_layout(
        width=600, 
        height=500, 
        xaxis=dict(tickfont=dict(size=35), title='', showgrid=False,), 
        yaxis=dict(tickfont=dict(size=30), title='', showgrid=False, ),
        #title=f'skew_inforate_{fmin}-{fmax}Hz_comb{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}',
        title = '', 
        showlegend=False,
        margin=dict(l=10, r=10, t=10, b=10, ),
    )
    pio.write_image(fig, f'{folder_dir}/shannon_inforate/skew_inforate_{int_chnl}_{list_freqrange[r][0]}-{list_freqrange[r][1]}Hz_sampfreq{sampfreq}_winsecond{list_winsize[r]}.png')
    fig.show()
    
    
    
    fig = go.Figure()
    fig.add_trace(go.Box(
        y=alz_group_kur, 
        boxmean=True, 
        name='AD',
    ))
    fig.add_trace(go.Box(
        y=con_group_kur,
        boxmean=True, 
        name='HC',
    ))
    fig.add_trace(go.Box(
        y=dem_group_kur, 
        boxmean=True, 
        name='FT',
    ))
    fig.update_layout(
        width=600, 
        height=500, 
        xaxis=dict(tickfont=dict(size=35), title='', showgrid=False,), 
        yaxis=dict(tickfont=dict(size=30), title='', showgrid=False, ),
        #title=f'kur_inforate_{fmin}-{fmax}Hz_comb{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}',
        title = '', 
        showlegend=False,
        margin=dict(l=10, r=10, t=10, b=10, ),
    )
    pio.write_image(fig, f'{folder_dir}/shannon_inforate/kur_inforate_{int_chnl}_{list_freqrange[r][0]}-{list_freqrange[r][1]}Hz_sampfreq{sampfreq}_winsecond{list_winsize[r]}.png')
    fig.show()
    
    
    
    # plot SECOND time to replace with the significant difference
    #quartile of the data 
    alz_mean_quan3 = np.quantile(alz_group_mean, 0.75)
    alz_var_quan3 = np.quantile(alz_group_var, 0.75)
    alz_skew_quan3 = np.quantile(alz_group_skew, 0.75)
    alz_kur_quan3 = np.quantile(alz_group_kur, 0.75)
    
    con_mean_quan3 = np.quantile(con_group_mean, 0.75)
    con_var_quan3 = np.quantile(con_group_var, 0.75)
    con_skew_quan3 = np.quantile(con_group_skew, 0.75)
    con_kur_quan3 = np.quantile(con_group_kur, 0.75)
    
    dem_mean_quan3 = np.quantile(dem_group_mean, 0.75)
    dem_var_quan3 = np.quantile(dem_group_var, 0.75)
    dem_skew_quan3 = np.quantile(dem_group_skew, 0.75)
    dem_kur_quan3 = np.quantile(dem_group_kur, 0.75)
    
    temp_group_index = ['Alzheimer', 'control', 'frontotemporal']
    
    print(max(alz_mean_quan3, con_mean_quan3, dem_mean_quan3))
    
    if kruskal_pval_moment1 < 0.05:
        fig = go.Figure()
        fig.add_trace(go.Box(
            y=alz_group_mean, 
            boxmean=True, 
            name='AD',
        ))
        fig.add_trace(go.Box(
            y=con_group_mean,
            boxmean=True, 
            name='HC',
        ))
        fig.add_trace(go.Box(
            y=dem_group_mean, 
            boxmean=True, 
            name='FT',
        ))
        fig.update_layout(
            width=600, 
            height=500, 
            xaxis=dict(tickfont=dict(size=35), title='', showgrid=False,), 
            yaxis=dict(tickfont=dict(size=30), title='', showgrid=False, ),
            #title=f'mean_inforate_{fmin}-{fmax}Hz_comb{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}',
            title = '', 
            showlegend=False,
            margin=dict(l=10, r=10, t=10, b=10, ),
        )
        xcoor_group = np.where(dunn_test_moment1_data[f'{fmin}-{fmax}']<0.05)[0]
        ycoor_group = np.where(dunn_test_moment1_data[f'{fmin}-{fmax}']<0.05)[1]
        print(xcoor_group, ycoor_group)
        for xc in xcoor_group:
            for yc in ycoor_group:
                # Determine the maximum y coordinate for the line
                if (xc, yc) == (0, 1):
                    y_line = max(np.quantile(alz_group_mean, 0.75), np.quantile(con_group_mean, 0.75))
                elif (xc, yc) == (0, 2):
                    y_line = max(np.quantile(alz_group_mean, 0.75), np.quantile(dem_group_mean, 0.75))
                elif (xc, yc) == (1, 2):
                    y_line = max(np.quantile(con_group_mean, 0.75), np.quantile(dem_group_mean, 0.75))
                else:
                    continue  # Skip if the condition is not met

                # Draw the line
                fig.add_shape(
                    type='line', 
                    x0=xc, x1=yc, 
                    y0=y_line, y1=y_line,
                    line=dict(color='black', width=1)
                )

                # Add a significance bracket (example between Group 1 and Group 2)
                fig.add_annotation(
                    x=0.5*(xc+yc),  # Midpoint between Group 1 and Group 2
                    y=y_line,  # Y position above the boxes
                    text="* p < 0.05",  # Significance label
                    showarrow=False,
                    font=dict(size=30),
                )
        pio.write_image(fig, f'{folder_dir}/shannon_inforate/mean_inforate_{int_chnl}_{list_freqrange[r][0]}-{list_freqrange[r][1]}Hz_sampfreq{sampfreq}_winsecond{list_winsize[r]}.png')
        fig.show()
    
    
    
    if kruskal_pval_moment2 < 0.05:
        fig = go.Figure()
        fig.add_trace(go.Box(
            y=alz_group_var, 
            boxmean=True, 
            name='AD',
        ))
        fig.add_trace(go.Box(
            y=con_group_var,
            boxmean=True, 
            name='HC',
        ))
        fig.add_trace(go.Box(
            y=dem_group_var, 
            boxmean=True, 
            name='FT',
        ))
        fig.update_layout(
            width=600, 
            height=500, 
            xaxis=dict(tickfont=dict(size=35), title='', showgrid=False,), 
            yaxis=dict(tickfont=dict(size=30), title='', showgrid=False, ),
            #title=f'var_inforate_{fmin}-{fmax}Hz_comb{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}',
            title = '', 
            showlegend=False,
            margin=dict(l=10, r=10, t=10, b=10, ),
        )
        xcoor_group = np.where(dunn_test_moment2_data[f'{fmin}-{fmax}']<0.05)[0]
        ycoor_group = np.where(dunn_test_moment2_data[f'{fmin}-{fmax}']<0.05)[1]
        print(xcoor_group, ycoor_group)
        for xc in xcoor_group:
            for yc in ycoor_group:
                # Determine the maximum y coordinate for the line
                if (xc, yc) == (0, 1):
                    y_line = max(np.quantile(alz_group_var, 0.75), np.quantile(con_group_var, 0.75))
                elif (xc, yc) == (0, 2):
                    y_line = max(np.quantile(alz_group_var, 0.75), np.quantile(dem_group_var, 0.75))
                elif (xc, yc) == (1, 2):
                    y_line = max(np.quantile(con_group_var, 0.75), np.quantile(dem_group_var, 0.75))
                else:
                    continue  # Skip if the condition is not met

                # Draw the line
                fig.add_shape(
                    type='line', 
                    x0=xc, x1=yc, 
                    y0=y_line, y1=y_line,
                    line=dict(color='black', width=1)
                )

                # Add a significance bracket (example between Group 1 and Group 2)
                fig.add_annotation(
                    x=0.5*(xc+yc),  # Midpoint between Group 1 and Group 2
                    y=y_line,  # Y position above the boxes
                    text="* p < 0.05",  # Significance label
                    showarrow=False,
                    font=dict(size=30),
                )
        pio.write_image(fig, f'{folder_dir}/shannon_inforate/var_inforate_{int_chnl}_{list_freqrange[r][0]}-{list_freqrange[r][1]}Hz_sampfreq{sampfreq}_winsecond{list_winsize[r]}.png')
        fig.show()
    
    
    if kruskal_pval_moment3 < 0.05:
        fig = go.Figure()
        fig.add_trace(go.Box(
            y=alz_group_skew, 
            boxmean=True, 
            name='AD',
        ))
        fig.add_trace(go.Box(
            y=con_group_skew,
            boxmean=True, 
            name='HC',
        ))
        fig.add_trace(go.Box(
            y=dem_group_skew, 
            boxmean=True, 
            name='FT',
        ))
        fig.update_layout(
            width=600, 
            height=500, 
            xaxis=dict(tickfont=dict(size=35), title='', showgrid=False,), 
            yaxis=dict(tickfont=dict(size=30), title='', showgrid=False, ),
            #title=f'skew_inforate_{fmin}-{fmax}Hz_comb{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}',
            title = '', 
            showlegend=False,
            margin=dict(l=10, r=10, t=10, b=10, ),
        )
        xcoor_group = np.where(dunn_test_moment3_data[f'{fmin}-{fmax}']<0.05)[0]
        ycoor_group = np.where(dunn_test_moment3_data[f'{fmin}-{fmax}']<0.05)[1]
        print(xcoor_group, ycoor_group)
        for xc in xcoor_group:
            for yc in ycoor_group:
                # Determine the maximum y coordinate for the line
                if (xc, yc) == (0, 1):
                    y_line = max(np.quantile(alz_group_skew, 0.75), np.quantile(con_group_skew, 0.75))
                elif (xc, yc) == (0, 2):
                    y_line = max(np.quantile(alz_group_skew, 0.75), np.quantile(dem_group_skew, 0.75))
                elif (xc, yc) == (1, 2):
                    y_line = max(np.quantile(con_group_skew, 0.75), np.quantile(dem_group_skew, 0.75))
                else:
                    continue  # Skip if the condition is not met

                # Draw the line
                fig.add_shape(
                    type='line', 
                    x0=xc, x1=yc, 
                    y0=y_line, y1=y_line,
                    line=dict(color='black', width=1)
                )

                # Add a significance bracket (example between Group 1 and Group 2)
                fig.add_annotation(
                    x=0.5*(xc+yc),  # Midpoint between Group 1 and Group 2
                    y=y_line,  # Y position above the boxes
                    text="* p < 0.05",  # Significance label
                    showarrow=False,
                    font=dict(size=30),
                )
        pio.write_image(fig, f'{folder_dir}/shannon_inforate/skew_inforate_{int_chnl}_{list_freqrange[r][0]}-{list_freqrange[r][1]}Hz_sampfreq{sampfreq}_winsecond{list_winsize[r]}.png')
        fig.show()
    
    
    
    if kruskal_pval_moment4 < 0.05:
        fig = go.Figure()
        fig.add_trace(go.Box(
            y=alz_group_kur, 
            boxmean=True, 
            name='AD',
        ))
        fig.add_trace(go.Box(
            y=con_group_kur,
            boxmean=True, 
            name='HC',
        ))
        fig.add_trace(go.Box(
            y=dem_group_kur, 
            boxmean=True, 
            name='FT',
        ))
        fig.update_layout(
            width=600, 
            height=500, 
            xaxis=dict(tickfont=dict(size=35), title='', showgrid=False,), 
            yaxis=dict(tickfont=dict(size=30), title='', showgrid=False, ),
            #title=f'kur_inforate_{fmin}-{fmax}Hz_comb{int_chnl}_sampfreq{sampfreq}_winsecond{int_win}_sldsecond{int_sld}',
            title = '', 
            showlegend=False,
            margin=dict(l=10, r=10, t=10, b=10, ),
        )
        xcoor_group = np.where(dunn_test_moment4_data[f'{fmin}-{fmax}']<0.05)[0]
        ycoor_group = np.where(dunn_test_moment4_data[f'{fmin}-{fmax}']<0.05)[1]
        print(xcoor_group, ycoor_group)
        for xc in xcoor_group:
            for yc in ycoor_group:
                # Determine the maximum y coordinate for the line
                if (xc, yc) == (0, 1):
                    y_line = max(np.quantile(alz_group_kur, 0.75), np.quantile(con_group_kur, 0.75))
                elif (xc, yc) == (0, 2):
                    y_line = max(np.quantile(alz_group_kur, 0.75), np.quantile(dem_group_kur, 0.75))
                elif (xc, yc) == (1, 2):
                    y_line = max(np.quantile(con_group_kur, 0.75), np.quantile(dem_group_kur, 0.75))
                else:
                    continue  # Skip if the condition is not met

                # Draw the line
                fig.add_shape(
                    type='line', 
                    x0=xc, x1=yc, 
                    y0=y_line, y1=y_line,
                    line=dict(color='black', width=1)
                )

                # Add a significance bracket (example between Group 1 and Group 2)
                fig.add_annotation(
                    x=0.5*(xc+yc),  # Midpoint between Group 1 and Group 2
                    y=y_line,  # Y position above the boxes
                    text="* p < 0.05",  # Significance label
                    showarrow=False,
                    font=dict(size=30),
                )
        pio.write_image(fig, f'{folder_dir}/shannon_inforate/kur_inforate_{int_chnl}_{list_freqrange[r][0]}-{list_freqrange[r][1]}Hz_sampfreq{sampfreq}_winsecond{list_winsize[r]}.png')
        fig.show()
    
    


#saving the data in .tsv format.
df_moment_info_alz_data = pd.DataFrame(np.squeeze(np.array(momentinfo_alz_data)).T, index=['moment1', 'moment2', 'moment3', 'moment4'], columns=[f'{j[0]}-{j[1]}' for j in list_freqrange])
df_moment_info_con_data = pd.DataFrame(np.squeeze(np.array(momentinfo_con_data)).T, index=['moment1', 'moment2', 'moment3', 'moment4'], columns=[f'{j[0]}-{j[1]}' for j in list_freqrange])
df_moment_info_dem_data = pd.DataFrame(np.squeeze(np.array(momentinfo_dem_data)).T, index=['moment1', 'moment2', 'moment3', 'moment4'], columns=[f'{j[0]}-{j[1]}' for j in list_freqrange])

os.makedirs(f'{folder_dir}/shannon_inforate/moment_inforate', exist_ok=True) # create the folder if the folder is not exist.
df_moment_info_alz_data.to_csv(f'{folder_dir}/shannon_inforate/moment_inforate/moment_inforate_alz_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}.tsv', sep='\t')
df_moment_info_con_data.to_csv(f'{folder_dir}/shannon_inforate/moment_inforate/moment_inforate_con_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}.tsv', sep='\t')
df_moment_info_dem_data.to_csv(f'{folder_dir}/shannon_inforate/moment_inforate/moment_inforate_dem_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}.tsv', sep='\t')



kruskal_test_moment1_data = pd.DataFrame(np.expand_dims(np.array(kruskal_test_moment1_data), axis=0), index=[f'{int_chnl}_moment1'], columns=[f'{j[0]}-{j[1]}' for j in list_freqrange])
kruskal_test_moment2_data = pd.DataFrame(np.expand_dims(np.array(kruskal_test_moment2_data), axis=0), index=[f'{int_chnl}_moment2'], columns=[f'{j[0]}-{j[1]}' for j in list_freqrange])
kruskal_test_moment3_data = pd.DataFrame(np.expand_dims(np.array(kruskal_test_moment3_data), axis=0), index=[f'{int_chnl}_moment3'], columns=[f'{j[0]}-{j[1]}' for j in list_freqrange])
kruskal_test_moment4_data = pd.DataFrame(np.expand_dims(np.array(kruskal_test_moment4_data), axis=0), index=[f'{int_chnl}_moment4'], columns=[f'{j[0]}-{j[1]}' for j in list_freqrange])

os.makedirs(f'{folder_dir}/shannon_inforate/kruskal_moment_inforate', exist_ok=True) # create the folder if the folder is not exist.
kruskal_test_moment1_data.to_csv(f'{folder_dir}/shannon_inforate/kruskal_moment_inforate/kruskal_moment1_inforate_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}.tsv', sep='\t')
kruskal_test_moment2_data.to_csv(f'{folder_dir}/shannon_inforate/kruskal_moment_inforate/kruskal_moment2_inforate_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}.tsv', sep='\t')
kruskal_test_moment3_data.to_csv(f'{folder_dir}/shannon_inforate/kruskal_moment_inforate/kruskal_moment3_inforate_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}.tsv', sep='\t')
kruskal_test_moment4_data.to_csv(f'{folder_dir}/shannon_inforate/kruskal_moment_inforate/kruskal_moment4_inforate_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}.tsv', sep='\t')



np.save(f'{folder_dir}/shannon_inforate/kruskal_moment_inforate/dunn_moment1_inforate_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}',  dunn_test_moment1_data) #save the dictionary data in .npy file
np.save(f'{folder_dir}/shannon_inforate/kruskal_moment_inforate/dunn_moment2_inforate_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}',  dunn_test_moment2_data) #save the dictionary data in .npy file
np.save(f'{folder_dir}/shannon_inforate/kruskal_moment_inforate/dunn_moment3_inforate_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}',  dunn_test_moment3_data) #save the dictionary data in .npy file
np.save(f'{folder_dir}/shannon_inforate/kruskal_moment_inforate/dunn_moment4_inforate_{int_chnl}_sampfreq{sampfreq}_winsecond{list_winsize}',  dunn_test_moment4_data) #save the dictionary data in .npy file



# end